# 02 - RNN 序列建模

## 学习目标
- 理解 LSTM 和 GRU
- 掌握序列数据处理
- 时间序列预测实战

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

## 1. 生成时间序列数据

In [ ]:
# 生成正弦波数据
def generate_data(n_samples=1000, seq_length=50):
    t = np.linspace(0, 100, n_samples)
    data = np.sin(t) + 0.1 * np.random.randn(n_samples)
    
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    
    return np.array(X), np.array(y)

X, y = generate_data()
X = X.reshape(-1, X.shape[1], 1)  # (samples, timesteps, features)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

# 划分数据集
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

## 2. LSTM 模型

In [ ]:
model = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(50, 1)),
    layers.LSTM(32),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

## 3. 训练

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(patience=5)]
)

## 4. 预测与可视化

In [ ]:
predictions = model.predict(X_test)

plt.figure(figsize=(15, 5))
plt.plot(y_test[:100], label='真实值', alpha=0.7)
plt.plot(predictions[:100], label='预测值', alpha=0.7)
plt.title('时间序列预测')
plt.xlabel('时间步')
plt.ylabel('值')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. LSTM vs GRU

In [ ]:
models = {
    'LSTM': keras.Sequential([
        layers.LSTM(64, input_shape=(50, 1)),
        layers.Dense(1)
    ]),
    'GRU': keras.Sequential([
        layers.GRU(64, input_shape=(50, 1)),
        layers.Dense(1)
    ]),
    'Bidirectional LSTM': keras.Sequential([
        layers.Bidirectional(layers.LSTM(64), input_shape=(50, 1)),
        layers.Dense(1)
    ])
}

results = {}
for name, model in models.items():
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
    loss = model.evaluate(X_test, y_test, verbose=0)
    results[name] = loss
    print(f'{name}: MSE = {loss:.4f}')

## 小结
- LSTM 解决长序列梯度消失问题
- GRU 是 LSTM 的简化版本
- 双向 RNN 捕捉前后文信息